# Plot the model-generated growth curves on top of the experimental data

This file generates Figures 3e and 6 in the main text.

It requires the growth data and config files as input and will save the figures to the path if specified (change file paths below as needed). 

The code will plot the true growth curves for each admixture and the curves predicted by specified parameters in the file. The file is currently set up to reproduce Figures 3e and 6, plotting estimates for the nude (immunocompromised) mice as well as for each groups of parameters estimated for the B6 (immunocompetent) mice.

The file also prints the data associated with Supplementary Table S4.

## Prep

Load needed packages

In [ ]:
%matplotlib widget
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
import os
from estimator import Estimator

Close any figures generated from previous runs

In [ ]:
plt.close("all")

## Define data and user-input parameters

Define the data file, config file, and save files

In [ ]:
config = "config_tcell.json"

# If you don't want to save figures, set to None
nude_save_file = "figures/nude_growth_pred.svg"
b6_save_file = "figures/b6_growth_pred_{}.svg" # Space is for cluster name

Define the groups to plot and the titles to use for the plots

In [ ]:
groups_nude = ["Grp. B1 nude (100% C1)", "Grp. B2 nude (80% C1; 20% C11)", "Grp. B3 nude (50% C1; 50% C11)", "Grp. B4 nude (20% C1; 80% C11)", "Grp. B5 nude (100% C11)"]
groups_b6 = ["Grp. A1 B6 (100% C1)", "Grp. A2 B6 (80% C1; 20% C11)", "Grp. A3 B6 (50% C1; 50% C11)", "Grp. A4 B6 (20% C1; 80% C11)", "Grp. A5 B6 (100% C11)"]
titles = ["100% Proliferative", "80% Proliferative\n20% Invasive", "50% Proliferative\n50% Invasive", "20% Proliferative\n80% Invasive", "100% Invasive"]

Create Estimator object from the config file

In [ ]:
es = Estimator(config)

## Functions

Function to get the initial values of the C1 (proliferative) and C11 (invasive) populations given the group name

In [ ]:
def get_init(group):
    if "A1" in group or "B1" in group: return [1, 0]
    if "A2" in group or "B2" in group: return [0.8, 0.2]
    if "A3" in group or "B3" in group: return [0.5, 0.5]
    if "A4" in group or "B4" in group: return [0.2, 0.8]
    if "A5" in group or "B5" in group: return [0, 1]

Function to run the model

In [ ]:
def run_model(es, init, g1, g11, k, m, p, a, f, l, end_time):
    sol = es.run_before_t(init, g1, g11, k, m, p, end_time)
    t1 = sol[3][-1]
    sol = es.run_incr_t(sol, g1, g11, k, m, p, a, f, l, end_time)
    t2 = sol[3][-1]
    sol = es.run_dec_t(sol, g1, g11, k, m, p, a, f, l, end_time) 
    return sol, t1, t2

Function to plot the nude data and model predictions. Shows the plot in the notebook and saves if nude_save_file is set.

@param time_range : list of two values where first value is the minimum time to plot and second value is the maximum time to plot

@param g1 : growth rate for C1 (proliferative) subclone

@param g11 : growth rate for C11 (invasive) subclone

@param k : parameter $k$ for the effect of C11 (invasive subclone) on C11 (proliferative subclone)

@param m : parameter $m$ for the effect of C1 (proliferative subclone) on C1 (invasive subclone)

In [ ]:
def plot_nude(time_range, g1, g11, k, m):
    # Set additional model parameters
    dt = 0.01
    cutoff_0 = 0.001

    # Define a list to store the times at which the non-dominating subclone goes extinct (Table S4).
    # Values will start with percentage of proliferative = 100% and decrease to 0%
    extinct_list = [] # Time
    extinct_bin = []  # Binary indicating which subline has gone extinct: 0 for C1, 1 for C11

    # Change figure size at the last part of this line if needed. First value is width, second value is height.
    fig, axes = plt.subplots(nrows=1, ncols=5, sharex=True, sharey=True, figsize=(10, 3))
    
    # Loop through the nude groups
    for i in range(len(groups_nude)):
        # Extract the data for that group
        group_df = es.growth_df[es.growth_df["group"] == groups_nude[i]]
        # For each mouse in that group, plot its growth on the group-specific plot
        for mid in group_df["id"].unique():
            axes[i].plot(group_df[group_df["id"]==mid]["day"], group_df[group_df["id"]==mid]["size"], color="black", linewidth=1)
        # Annotate the group-specific plot
        axes[i].set(title=titles[i], ylim=[-1, max(es.growth_df["size"])+10], xlabel="Day", ylabel="scaled size")
        
        # Plot estimated data
        # Get the initial ratio of the subclones given the current group name
        init = get_init(groups_nude[i])
        # The initial T cell size is 0
        init += [0]
        
        # Run the model with the T cell-specific parameters equal to 0 (l is 1 to avoid divide by zero error)
        sol, t1, t2 = run_model(es, init, g1, g11, k, m, 0, 0, 0, 1, time_range[1])

        # Plot C1 (proliferative subclone) in orange
        axes[i].plot(sol[3], sol[0], c="orange", label="C1")
        # Plot C11 (invasive subclone) in blue
        axes[i].plot(sol[3], sol[1], c="blue", label="C11")

        # Print data for Supplementary Tables S4 and S5 -- comment if desired
        if (len(sol[3][sol[0]==0]) > 0): # C1 has gone extinct
            extinct_list += [round(sol[3][sol[0]==0][0], 2)]
            extinct_bin += [0]
            # print("Time at which C1 becomes 0:", sol[3][sol[0]==0][0])
        if (len(sol[3][sol[1]==0]) > 0):
            # print("Time at which C11 becomes 0:", sol[3][sol[1]==0][0])
            extinct_list += [round(sol[3][sol[1]==0][0], 2)]
            extinct_bin += [1]

    # Annotate the plot
    fig.suptitle("Immunocomprimised mice\ng1={}, g11={}, k={}, m={}".format(g1, g11, k, m))
    plt.tight_layout()
    # Save if file is provided
    if nude_save_file:
        plt.savefig(nude_save_file)
    # Show the figure
    plt.show()

    return extinct_list, extinct_bin
    

Function to plot the B6 (immunocomprimised) data and model predictions. Shows the plot in the notebook and saves if b6_save_file is set.

@param time_range : list of two values where first value is the minimum time to plot and second value is the maximum time to plot

@param t_init : initial size of the T cell population

@param g1 : growth rate for C1 (proliferative) subclone

@param g11 : growth rate for C11 (invasive) subclone

@param k : parameter $k$ for the effect of C11 (invasive subclone) on C11 (proliferative subclone)

@param m : parameter $m$ for the effect of C1 (proliferative subclone) on C1 (invasive subclone)

@param d : parameter $d$ for the effect of the T cells on C1 (proliferative subclone)

@param a : recruitment rate $a$ of the T cells by C1 (proliferative subclone)

@param l : Michaelis–Menten constant $l$ for the limiting factor on the recruitment of T cells

@param f : exhaustion/death rate $f$ of T cells

In [ ]:
def plot_b6(time_range, t_init, g1, g11, k, m, d, a, f, l):
    # Set additional model parameters
    dt = 0.01
    cutoff_0 = 0.001

    # Define a list to store the times at which the non-dominating subclone goes extinct (Table S4).
    # Values will start with percentage of proliferative = 100% and decrease to 0%
    extinct_list = [] # Time
    extinct_bin = []  # Binary indicating which subline has gone extinct: 0 for C1, 1 for C11

    # Define the maximum value of the y-axis to be 100 mm^3 larger than the maximum value across all B6 groups
    ylim = es.growth_df[es.growth_df["group"].isin(groups_b6)]["size"].max()

    # Change figure size at the last part of this line if needed. First value is width, second value is height.
    fig, axes = plt.subplots(nrows=2, ncols=5, sharex=True, height_ratios=[0.75, 0.25], figsize=(10,4))

    # Loop through the B6 groups
    for i in range(len(groups_b6)):
        # Extract the data for that group
        group_df = es.growth_df[es.growth_df["group"] == groups_b6[i]]
        # For each mouse in that group, plot its growth on the group-specific plot
        for mid in group_df["id"].unique():
            axes[0][i].plot(group_df[group_df["id"]==mid]["day"], group_df[group_df["id"]==mid]["size"], color="black", linewidth=1)

        # Plot estimated data
        # Get the initial ratio of the subclones given the current group name
        init = get_init(groups_b6[i])
        # Add T cell initial size to the inital value list
        init += [t_init]
        
        # Run the model
        sol, t1, t2 = run_model(es, init, g1, g11, k, m, d, a, f, l, time_range[1])     

        # Print data for Supplementary Tables S4 and S5 -- comment if desired
        if (len(sol[3][sol[0]==0]) > 0): # C1 has gone extinct
            extinct_list += [round(sol[3][sol[0]==0][0], 2)]
            extinct_bin += [0]
            # print("Time at which C1 becomes 0:", sol[3][sol[0]==0][0])
        if (len(sol[3][sol[1]==0]) > 0):
            # print("Time at which C11 becomes 0:", sol[3][sol[1]==0][0])
            extinct_list += [round(sol[3][sol[1]==0][0], 2)]
            extinct_bin += [1]
        
        # Plot the model-generated curves with C1 (proliferative) in orange and C11 (invasive) in blue
        # Plot the winning subline second so it is in front of the other subline
        if sol[0][-1] > sol[1][-1]: 
            axes[0][i].plot(sol[3], sol[1], color="blue", label="C11")
            axes[0][i].plot(sol[3], sol[0], color="orange", label="C1")
            color = "orange"
        else:
            axes[0][i].plot(sol[3], sol[0], color="orange", label="C1")
            axes[0][i].plot(sol[3], sol[1], color="blue", label="C11")
            color = "blue"
       
        # Plot the T cell population on the lower plot
        axes[1][i].plot(sol[3], sol[2], color="magenta", label="T")

        # Annotate the group-specific plot            
        axes[0][i].set(title=titles[i], ylim=[-1, ylim])

    # Annotate the whole plot
    fig.suptitle("Immunocompetent mice\ng1={}, g11={}, k={}, m={}, d={}, a={}, f={}, l={}".format(g1, g11, k, m, d, a, f, l))
    plt.tight_layout()

    if b6_save_file:
        plt.savefig(b6_save_file)

    # Show the figure
    plt.show()

    return extinct_list, extinct_bin

## Plotting

Define the parameters for the nude model and plot the predictions

In [ ]:
g1 = 0.135  # Growth rate of C1 (proliferative subclone)
g11 = 0.107 # Growth rate of C11 (invasive subclone)
time_range = [0, 50]  # Time range to run the simulation

Plot the nude model-generated curves

In [ ]:
# Define range of m and k values to use
ms = [-0.02, -0.14, -0.2]  # Effect of C1 (proliferative subclone) on C11 (invasive subclone)
ks = [-0.7051*m + 0.0086 for m in ms] # Effect of C11 (invasive subclone) on C1 (proliferative subclone) as defined by previously estimated linear relationship

# Lists to hold all the extinction information
extinct_df = []
extinct_bin_df = []

for i in range(len(ms)):
    extinct_list, extinct_bin = plot_nude(time_range, g1, g11, ks[i], ms[i])
    # Add extinction and size information for this m value to our lists
    extinct_df += [extinct_list]
    extinct_bin_df += [extinct_bin]

# Turn extinction information into data frames
extinct_df = pd.DataFrame(extinct_df, columns=[100, 80, 50, 20, 0], index=ms)
extinct_bin_df = pd.DataFrame(extinct_bin_df, columns=[100, 80, 50, 20, 0], index=ms)

# Print extinction information
print("Time at which it takes the non-dominating subclone to go extinct")
print(extinct_df)
print("Which subclone goes extinct: 0 = C1 (proliferative), 1 = C11 (invasive)")
print(extinct_bin_df)

## B6 (immunocompetent) plots

Define the values of general parameters and the parameters for each of the T cell model clusters

In [ ]:
g1 = 0.135  # Growth rate of C1 (proliferative subclone)
g11 = 0.107 # Growth rate of C11 (invasive subclone)
t_init = 0.001 # Initial T cell population size; 0.001
time_range = [0,125] # Time range to run the simulations

groups = {
    "1a": {"d":-1.0, "m":-0.062857, "k":0.052486, "a":1.616667, "f":1.116667, "af": 1.818147, "l":0.000550},
    "1b": {"d":-5.0, "m":-0.061487, "k":0.051510, "a":1.501582, "f":1.201582, "af": 1.466692, "l": 0.000545},
    "2a": {"d":-7.824412, "m":-0.090987, "k":0.072528, "a":1.783144, "f":1.483144, "af": 1.222601, "l":0.000549}, # smaller chunk}
    "2b": {"d":-5.969474, "m":-0.140418, "k":0.107748, "a":2.199812, "f":1.684158, "af": 1.309783, "l":0.000553} # bigger chunk
}

Plot the model-generated curves for each cluster's parameters

In [ ]:
# Lists to hold all the extinction and size information
extinct_df = []
extinct_bin_df = []

# Loop through the groups and plot each
for g in groups.keys():
    
    # Plot data and model-generated curves
    extinct_list, extinct_bin = plot_b6(time_range, t_init, g1, g11, groups[g]["k"], groups[g]["m"], groups[g]["d"], groups[g]["a"], groups[g]["f"], groups[g]["l"])
    # Add title
    plt.suptitle("Group {}".format(g))

    # Save
    if b6_save_file:
        plt.savefig(b6_save_file.format(g))

    # Add information from this run to the lists
    extinct_df += [extinct_list]
    extinct_bin_df += [extinct_bin]

# Turn extinction and size data into data frames
extinct_df = pd.DataFrame(extinct_df, columns=[100, 80, 50, 20, 0], index=["1a", "1b", "2a", "2b"])
extinct_bin_df = pd.DataFrame(extinct_bin_df, columns=[100, 80, 50, 20, 0], index=["1a", "1b", "2a", "2b"])

# Print extinction and size information
print("Time at which it takes the non-dominating subclone to go extinct")
print(extinct_df)
print("Which subclone goes extinct: 0 = C1 (proliferative), 1 = C11 (invasive)")
print(extinct_bin_df)